In [1]:
import os
import astropy.units as u
from astropy.coordinates import FK5, SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astroquery.astrometry_net import AstrometryNet

import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import simple_norm
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales

import io
import requests
import astropy.units as u
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

import numpy as np
from joblib import Parallel, delayed

from astroquery.sdss import SDSS
from astropy import coordinates as coords
import pandas as pd
from astropy.coordinates import SkyCoord, FK5
import astropy.units as u
import matplotlib
matplotlib.use('Agg')


import io
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import wcsaxes
import requests

In [5]:
GEHR_list = pd.read_csv('FinalGEHRList.csv',index_col=False)
aux_host = pd.read_csv('aux_host.csv',index_col=False)
host_galaxies = np.unique(GEHR_list['GALAXY'])
url = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits"

In [15]:
search_status = 'galaxy_match_status.txt'
if not os.path.exists(search_status):
    print(f"Archivo no encontrado. Creando {search_status} para guardar valores... \n")
    with open(search_status, 'w') as f:
        pass




for host in host_galaxies:

    status_list = []
    with open(search_status, 'r') as f:
        for linea in f:
            status_list.append(linea.strip())

    if f"{host}" not in status_list:

        print(f"{host}: \n" )
        selec = GEHR_list[GEHR_list['GALAXY'] == host]
        host_info_aux = aux_host[aux_host['GALAXY'] == host]
        print(f"{host_info_aux.iloc[0]['SURVEY']} , {host_info_aux.iloc[0]['RA']} , {host_info_aux.iloc[0]['DEC']} \n" )
        for i in range(len(selec)):
            print(f"{selec.iloc[i]['CLASS']}	{selec.iloc[i]['RA']}	{selec.iloc[i]['DEC']} \n")
        #host_coordinates = SkyCoord(
        #    ra=host_info_aux.iloc[0]['RA'], 
        #    dec=host_info_aux.iloc[0]['DEC'], 
        #    unit=(u.hourangle, u.deg), # RA en dec, DEC en dec
        #    frame='icrs', 
        #    equinox='J2000'
        #)
        ra_real = host_info_aux.iloc[0]['RA'] #host_coordinates.ra.deg          
        dec_real = host_info_aux.iloc[0]['DEC'] #host_coordinates.dec.deg                
        fov_en_grados = (selec.iloc[0]['FOV(deg)'] * u.deg).value

        img_color = None
        encuesta_usada = "Ninguna"

        params = {
            "hips": host_info_aux.iloc[0]['SURVEY'],
            "ra": ra_real,
            "dec": dec_real,
            "fov": fov_en_grados,
            "width": 2000,
            "height": 2000,
            "projection": "TAN",
            "coordsys": "icrs",
            "format": "fits"
        }




        response = requests.get(url, params=params)
        #img_color = Image.open(io.BytesIO(response.content))
        #im2 = ax.imshow(img_color)
        #ax.axis("off")

        #fig.savefig(f"{host_info_aux.iloc[0]['NUMERO']}.{host_info_aux.iloc[0]['GALAXY']}.png")

        with fits.open(io.BytesIO(response.content)) as hdul:
            header = hdul[0].header
            wcs_imagen = WCS(header).celestial
            datos_crudos = hdul[0].data 
            datos_reordenados = np.moveaxis(datos_crudos, 0, -1)
            datos_imagen = datos_reordenados[:, :, :3]

        


        fig = plt.figure(figsize=(15, 15))

        ax = fig.add_subplot(1, 1, 1, projection=wcs_imagen)
        im = ax.imshow(datos_imagen, origin='lower') 
        ax.coords[0].set_axislabel('Ascensión Recta (RA)')
        ax.coords[1].set_axislabel('Declinación (DEC)')
        ax.coords.grid(color='white', alpha=0.5, linestyle='solid')



        f_px,f_py,RA_vG,DEC_vG = [],[],[],[]
        for i in range(len(selec)):
            Point = f"{selec['RA'].iloc(0)[i]} {selec['DEC'].iloc(0)[i]}"
            coord = SkyCoord(Point, unit=(u.deg, u.deg), frame="icrs")
            RA_vG.append(coord.ra)
            DEC_vG.append(coord.dec)
            x, y = coord.to_pixel(wcs_imagen, origin=0)
            f_px.append(x)
            f_py.append(y)
        ax.plot(f_px, f_py, fillstyle='none',
                color='red', marker='o', markersize=13, linewidth = 0,
                label='GEHR')

        labels = np.array(selec['CLASS'])
        for i, label in enumerate(labels):

            color_annotate = 'black'
            ha_annotate = 'center'

            if 'new' in label:
                 color_annotate = 'orange'
                 ha_annotate = 'left'
            if 'old' in label:
                 color_annotate = 'blue'
                 ha_annotate = 'right'
            if 'unid' in label:
                 color_annotate = 'red'
                 ha_annotate = 'center'

            ax.annotate(
                label,  # The text label
                (f_px[i], f_py[i]),  # The point being annotated (xy)
                textcoords="offset points",  # How to position the text
                xytext=(0, 10),  # Distance from the point (x, y)
                color=color_annotate,  # Text color
                ha=ha_annotate,  # Horizontal alignment
                bbox=dict(
                    boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.4
                ),  # Fondo blanco
            )
        

        # Busqueda de espectros de SDSS

        ancho_px = header['NAXIS1']
        alto_px = header['NAXIS2']
        escala_x = abs(header['CDELT1']) 
        escala_y = abs(header['CDELT2'])
        ancho_arcsec = ancho_px * escala_x * 3600
        alto_arcsec = alto_px * escala_y * 3600
        centro_x = (header['NAXIS1'] - 1) / 2.0
        centro_y = (header['NAXIS2'] - 1) / 2.0
        ra_cent, dec_cent = wcs_imagen.all_pix2world(centro_x, centro_y,0)
        center_im = SkyCoord(ra_cent,dec_cent, unit = (u.deg,u.deg),frame = 'icrs')

        data_releases = [17,18,19]
        dr_colors = ["#fd7af9c6","#7afc00c5","#ff7b00c5"]
        dr_tab_pos = ["upper left","lower left","lower right"]
        dr_markersize = [18,14,10]


        for DR_SDSS in range(len(data_releases)):
            xid_spec = SDSS.query_region(center_im, height = f'{alto_arcsec} arcsec', width = f'{ancho_arcsec} arcsec', spectro=True,data_release=data_releases[DR_SDSS])
            s_x,s_y,RA_vS,DEC_vS = [],[],[],[]
            if type(xid_spec) != type(None):
                    for i in range(len(xid_spec)):
                            coord_p = SkyCoord(xid_spec['ra'][i],xid_spec['dec'][i], unit=(u.deg, u.deg), frame='icrs')
                            RA_vS.append(coord_p.ra)
                            DEC_vS.append(coord_p.dec)
                            x, y = coord_p.to_pixel(wcs_imagen, origin=0)
                            s_x.append(x)
                            s_y.append(y)
            separation = 5
            catalog_matches = []
            if type(xid_spec) != type(None):
                    ax.plot(s_x, s_y,
                            markerfacecolor=dr_colors[DR_SDSS], marker='X', markersize=dr_markersize[DR_SDSS], linewidth = 0, markeredgewidth = 1,markeredgecolor='white',
                            label=f"SDSS spectra DR{data_releases[DR_SDSS]}",alpha=0.5)
                    sdss_view = SkyCoord(ra=RA_vS, dec=DEC_vS)
                    GEHR_view = SkyCoord(ra=RA_vG, dec=DEC_vG)
                    idx, d2d, d3d = GEHR_view.match_to_catalog_sky(sdss_view)
                    separation = separation
                    max_sep = separation * u.arcsec
                    sep_constraint = d2d < max_sep
                    c_matches = GEHR_view[sep_constraint]
                    catalog_matches = sdss_view[idx[sep_constraint]]
            if len(catalog_matches) >= 1:
                    data_table_onscreen = []
                    col_labels_tab = ['RA','DEC','PLATE', 'MJD','FIBER']
                    row_labels_tab = []
                    for b in range(len(idx)):
                        if d2d[b] < max_sep:
                            tmp_row = []
                            center_match = SkyCoord(sdss_view[idx[b]].ra,sdss_view[idx[b]].dec, unit = (u.deg,u.deg),frame = 'icrs')
                            xid_match = SDSS.query_region(center_match, height = f'{separation} arcsec', width = f'{separation} arcsec', spectro=True,data_release=data_releases[DR_SDSS])
                            row_labels_tab.append(selec['NOMBRE'].iloc[b])
                            tmp_row.append(xid_match['ra'][0])
                            tmp_row.append(xid_match['dec'][0])
                            tmp_row.append(xid_match['plate'][0])
                            tmp_row.append(xid_match['mjd'][0])
                            tmp_row.append(xid_match['fiberID'][0])
                            data_table_onscreen.append(tmp_row)
                    print(row_labels_tab)
                    print(data_table_onscreen)
                    ax.axis('on')
                    table = ax.table(cellText=data_table_onscreen,
                                    colLabels=col_labels_tab,
                                    rowLabels=row_labels_tab,
                                    colWidths=[0.17, 0.17,0.06,0.06,0.05],
                                    loc=f"{dr_tab_pos[DR_SDSS]}") # 'loc' specifies the table's position

                    table.auto_set_font_size(False)
                    table.set_fontsize(10)
                    table.scale(0.8, 1.0) # Scale the table size


        ax.legend(title = f"{host_info_aux.iloc[0]['GALAXY']} Max sep {separation} arcsec")
        print(f"¡Éxito! Imagen FITS obtenida. \n")
        with open(search_status, 'a') as f:
            f.write(f"{host_info_aux.iloc[0]['GALAXY']}\n")

        fig.savefig(f"{host_info_aux.iloc[0]['NUMERO']}.{host_info_aux.iloc[0]['GALAXY']}.png")


        

Archivo no encontrado. Creando galaxy_match_status.txt para guardar valores... 

IC0010: 

CDS/P/DSS2/color , 5.07205 , 59.30385 

A_old	5.1125	59.2913888888889 

B_old	5.07083333333333	59.3094444444444 

¡Éxito! Imagen FITS obtenida. 

IC2574: 

CDS/P/DESI-Legacy-Surveys/DR10/color , 157.097278 , 68.412159 

A_unid	157.09728	68.41216 

B_unid	157.09728	68.41216 

C_unid	157.09728	68.41216 

¡Éxito! Imagen FITS obtenida. 

M101: 

CDS/P/SDSS9/color-alt , 210.80227 , 54.34895 

A_old	210.6167	54.2758 

B_old	210.7542	54.2414 

C_old	210.9208	54.3172 

D_old	210.9708	54.3683 

E_old	211.1208	54.3981 



¡Éxito! Imagen FITS obtenida. 

M33: 

CDS/P/SDSS9/color-alt , 23.46204 , 30.66022 

A_old	23.1917	30.6475 

B_old	23.3	30.645 

C_old	23.3917	30.6917 

D_old	23.6375	30.785 

¡Éxito! Imagen FITS obtenida. 

M66: 

CDS/P/SDSS9/color-alt , 170.06254 , 12.99161 

A_new	170.055622	13.0071813 

B_new	170.077213	12.9896862 

C_new	170.049079	12.9883075 

['M66B_new']
[[170.078514674638, 12.9898741640805, 1605, 53062, 452]]
['M66B_new']
[[170.078514674638, 12.9898741640805, 1605, 53062, 452]]
['M66B_new']
[[170.078514674638, 12.9898741640805, 1605, 53062, 452]]
¡Éxito! Imagen FITS obtenida. 

M81: 

CDS/P/SDSS9/color-alt , 148.88822 , 69.06529 

A_old	148.9708	68.9842 

B_old	148.7375	69.1467 

¡Éxito! Imagen FITS obtenida. 

M96: 

CDS/P/SDSS9/color-alt , 161.69035 , 11.82 

A_new	161.7042	11.8128 

¡Éxito! Imagen FITS obtenida. 

MRK116: 

CDS/P/SDSS9/color-alt , 143.50845 , 55.24107 

A_old	143.5083	55.2411 

['MRK116A_old']
[[143.508473305155, 55.2410533686072, 555, 52266, 558]]
['MRK116

¡Éxito! Imagen FITS obtenida. 

NGC2366: 

CDS/P/DESI-Legacy-Surveys/DR10/color , 112.22542 , 69.21497 

A_old	112.125	69.1936 

B_old	112.1958	69.1908 

C_old	112.1792	69.1897 

D_old	112.2292	69.2158 

¡Éxito! Imagen FITS obtenida. 

NGC2403: 

CDS/P/SDSS9/color-alt , 114.21374 , 65.60268 

A_old	114.1917	65.6169 

B_old	114.0833	65.6178 

C_old	114.2792	65.6108 

¡Éxito! Imagen FITS obtenida. 

NGC2500: 

CDS/P/SDSS9/color-alt , 120.47154 , 50.73714 

A_new	120.480955	50.7440822 

B_new	120.444786	50.7339321 

C_new	120.45	50.7339 

D_new	120.4792	50.7453 

¡Éxito! Imagen FITS obtenida. 

NGC2541: 

CDS/P/DESI-Legacy-Surveys/DR10/color , 123.66728 , 49.06193 

A_old	123.7	49.0664 

B_old	123.6542	49.0497 

C_old	123.6542	49.0647 

A_new	123.6542	49.0497 

B_new	123.6542	49.0647 

C_new	123.6458	49.0536 

D_new	123.6833	49.0353 

['NGC2541A_old', 'NGC2541B_old', 'NGC2541A_new']
[[123.698033477002, 49.0668691926314, 440, 51912, 134], [123.655391655199, 49.0500114442075, 440, 51912, 13

¡Éxito! Imagen FITS obtenida. 

NGC3198: 

CDS/P/DESI-Legacy-Surveys/DR10/color , 154.97918 , 45.54982 

A_old	154.9417	45.5175 

A_new	154.9625	45.5467 

B_new	154.9417	45.5172 

C_new	154.9917	45.5406 

['NGC3198A_old', 'NGC3198B_new']
[[154.94326410509, 45.5179982050633, 944, 52614, 162], [154.94326410509, 45.5179982050633, 944, 52614, 162]]
['NGC3198A_old', 'NGC3198B_new']
[[154.94326410509, 45.5179982050633, 944, 52614, 162], [154.94326410509, 45.5179982050633, 944, 52614, 162]]
['NGC3198A_old', 'NGC3198B_new']
[[154.94326410509, 45.5179982050633, 944, 52614, 162], [154.94326410509, 45.5179982050633, 944, 52614, 162]]
¡Éxito! Imagen FITS obtenida. 

NGC3319: 

CDS/P/SDSS9/color-alt , 159.78939 , 41.68669 

A_old	159.7667	41.6614 

B_old	159.75	41.6689 

C_old	159.825	41.7019 

A_new	159.75	41.6686 

B_new	159.7667	41.6608 

C_new	159.7833	41.6672 



['NGC3319A_old', 'NGC3319B_old', 'NGC3319C_old', 'NGC3319A_new', 'NGC3319B_new']
[[159.766859891927, 41.6613689225676, 1360, 53003, 637], [159.751582480511, 41.6690988747166, 1361, 53047, 221], [159.823391141316, 41.7019648237255, 1361, 53047, 222], [159.751582480511, 41.6690988747166, 1361, 53047, 221], [159.766859891927, 41.6613689225676, 1360, 53003, 637]]
['NGC3319A_old', 'NGC3319B_old', 'NGC3319C_old', 'NGC3319A_new', 'NGC3319B_new']
[[159.766859891927, 41.6613689225676, 1360, 53003, 637], [159.751582480511, 41.6690988747166, 1361, 53047, 221], [159.823391141316, 41.7019648237255, 1361, 53047, 222], [159.751582480511, 41.6690988747166, 1361, 53047, 221], [159.766859891927, 41.6613689225676, 1360, 53003, 637]]


['NGC3319A_old', 'NGC3319B_old', 'NGC3319C_old', 'NGC3319A_new', 'NGC3319B_new']
[[159.766859891927, 41.6613689225676, 1360, 53003, 637], [159.751582480511, 41.6690988747166, 1361, 53047, 221], [159.823391141316, 41.7019648237255, 1361, 53047, 222], [159.751582480511, 41.6690988747166, 1361, 53047, 221], [159.766859891927, 41.6613689225676, 1360, 53003, 637]]
¡Éxito! Imagen FITS obtenida. 

NGC3370: 

CDS/P/DESI-Legacy-Surveys/DR10/color , 161.76701 , 17.27378 

A_new	161.761545	17.2784372 

B_new	161.759358	17.2864263 

C_new	161.772788	17.2684047 

¡Éxito! Imagen FITS obtenida. 

NGC4214: 

CDS/P/SDSS9/color-alt , 183.91321 , 36.32689 

A_unid	183.91321	36.32689 

A_new	183.9125	36.3269 

B_new	183.9208	36.3192 

C_new	183.9208	36.3178 

¡Éxito! Imagen FITS obtenida. 

NGC4236: 

CDS/P/DSS2/color , 184.1755 , 69.462583 

A_unid	184.1755	69.46258 

B_unid	184.1755	69.46258 

¡Éxito! Imagen FITS obtenida. 

NGC4258: 

CDS/P/SDSS9/color-alt , 184.7396 , 47.30397 

A_old	184.7292	47.2794

¡Éxito! Imagen FITS obtenida. 

NGC4414: 

CDS/P/SDSS9/color-alt , 186.61312 , 31.22353 

A_new	186.4917	33.5269 

B_new	186.4833	33.5144 

C_new	186.4292	33.5136 

¡Éxito! Imagen FITS obtenida. 

NGC4496A: 

CDS/P/SDSS9/color-alt , 187.91333 , 3.93944 

A_new	187.9083	3.9306 

B_new	187.9167	3.9419 

C_new	187.8958	3.925 

¡Éxito! Imagen FITS obtenida. 

NGC4496B: 

CDS/P/SDSS9/color-alt , 187.920395 , 3.92635293 

A_new	187.9208	3.9208 

B_new	187.9125	3.925 

¡Éxito! Imagen FITS obtenida. 

NGC4535: 

CDS/P/SDSS9/color-alt , 188.58458 , 8.19792 

A_new	188.576204	8.2398179 

B_new	188.5708	8.2089 

['NGC4535A_new', 'NGC4535B_new']
[[188.576240467803, 8.23981305321984, 1627, 53473, 611], [188.5718266551, 8.20872891658758, 1627, 53473, 618]]
['NGC4535A_new', 'NGC4535B_new']
[[188.576240467803, 8.23981305321984, 1627, 53473, 611], [188.5718266551, 8.20872891658758, 1627, 53473, 618]]
['NGC4535A_new', 'NGC4535B_new']
[[188.576240467803, 8.23981305321984, 1627, 53473, 611], [188.57182665

¡Éxito! Imagen FITS obtenida. 

NGC5584: 

CDS/P/SDSS9/color-alt , 215.599 , -0.38767 

A_new	215.597418	-0.3739571 

B_new	215.5917	-0.3836 

C_new	215.5917	-0.3692 

¡Éxito! Imagen FITS obtenida. 

NGC6822: 

CDS/P/DSS2/color , 296.24042 , -14.80333 

A_unid	296.24042	-14.80333 

B_unid	296.24042	-14.80333 

A_new	296.272142	-14.7216777 

B_new	296.220008	-14.7199241 

¡Éxito! Imagen FITS obtenida. 

NGC7331: 

CDS/P/SDSS9/color-alt , 339.266927 , 34.415756 

A_new	339.292483	34.3666383 

B_new	339.277547	34.4385842 

C_new	339.256523	34.4213392 

D_new	339.261364	34.4042644 

E_new	339.278677	34.4143966 

¡Éxito! Imagen FITS obtenida. 

UGC08091: 

CDS/P/DESI-Legacy-Surveys/DR10/color , 194.66845 , 14.21739 

A_new	194.6625	14.2117 

B_new	194.6667	14.2158 

['UGC08091B_new']
[[194.667469463483, 14.2168800234195, 1771, 53498, 181]]
['UGC08091B_new']
[[194.667469463483, 14.2168800234195, 1771, 53498, 181]]
['UGC08091B_new']
[[194.667469463483, 14.2168800234195, 1771, 53498, 181]]
¡Éx